In [ ]:
import numpy as np
import astropy
from astropy.io import fits
import sep
import matplotlib.pyplot as plt
from matplotlib import rcParams

In [ ]:
%matplotlib inline  
rcParams['figure.figsize'] = [10.,8.]

In [ ]:
data = fits.getdata('./hlsp_hudf12_hst_wfc3ir_udfmain_f105w_v1.0_drz.fits')
data = np.astype(data,np.float32)

In [ ]:
m, s = np.mean(data), np.std(data)
plt.imshow(data, interpolation='nearest',cmap='grey',vmin=m-s,vmax=m+s,origin='lower')
_ = plt.colorbar()
plt.savefig('final_1_raw.png',bbox_inches='tight',dpi = 600)

Characterizing the background and noise values.


In [ ]:
bkg = sep.Background(data)
print(bkg.globalback)
print(bkg.globalrms)
bkg_image = bkg.back()

Showing and saving plot of the background data.

In [ ]:
plt.imshow(bkg_image, interpolation='nearest', cmap='gray', origin='lower')
_ = plt.colorbar()
plt.savefig('final_2_background.png',bbox_inches='tight',dpi = 600)

Evaluating the background noise as 2-d `numpy` array, same size as original image.


In [ ]:
bkg_rms = bkg.rms()

Showing and saving the background noise data.

In [ ]:
plt.imshow(bkg_rms, interpolation='nearest', cmap='gray', origin='lower')
plt.colorbar()
_ = plt.colorbar()
plt.savefig('final_3_noise.png',bbox_inches='tight',dpi = 600)

Subtracting the background data and noise from the origin image data. This facilitates the image extraction and avoids    accidentally extracting patches of noise as though they were objects. 

In [ ]:
data_sub = data - bkg

Using the background-subtracted image data in a call to `sep`'s `extract` function. The threshold is set at 1.5*sigma and will be dynamically adjust on a per-pixel basis with the noise values stored in `bkg.globalrms`. The object data is stored in a structered `numpy` array.

In [ ]:
objects = sep.extract(data_sub, 1.5, err=bkg.globalrms)

Getting a quick count of how many objects were detected.

In [ ]:
object_count = len(objects)
print(f'Found {object_count} objects.')

Plotting the background-subtracted image data in the same way as the raw data, then iterating through each object and uses the stored values for:
 - the x and y position,
 - the orthogonal major and minor axes of the object's enclosing ellipse,
 - and the angle to which those axes are rotated from the standard i,j basis
to plot ellipses on the image where the detected object lie. 

In [ ]:
from matplotlib.patches import Ellipse

# plot background-subtracted image
fig, ax = plt.subplots()
m, s = np.mean(data_sub), np.std(data_sub)
im = ax.imshow(data_sub, interpolation='nearest', cmap='gray',
               vmin=m-s, vmax=m+s, origin='lower')

# plot an ellipse for each object
for i in range(len(objects)):
    e = Ellipse(xy=(objects['x'][i], objects['y'][i]),
                width=6*objects['a'][i],
                height=6*objects['b'][i],
                angle=objects['theta'][i] * 180. / np.pi)
    e.set_facecolor('none')
    e.set_edgecolor('red')
    ax.add_artist(e)

plt.savefig('final_4_detection.png',bbox_inches='tight',dpi = 600)

Demonstrates the keys available to access the available fields of the structured array returned by `sep.extract`.

In [ ]:
objects.dtype.names

Computes the brightness of each of the detected objects using simple circular photometry. Instead of defining an outer annulus to get the background/noise value to subtract, uses the background object's global value. 

In [ ]:
flux, fluxerr, flag = sep.sum_circle(data_sub, objects['x'], objects['y'],
                                     3.0, err=bkg.globalrms, gain=1.0)

Show the calculated flux and error for the first ten objects.  
Plotting a histogram of the log of the flux values for f105.

In [ ]:
for i in range(10):
    print("object {:d}: flux = {:f} +/- {:f}".format(i, flux[i], fluxerr[i]))

flux_log = np.log(flux)

f,ax = plt.subplots(1,1,figsize = (10,7))
_ =ax.hist(flux_log,bins=50)

## What are the mean, median, and standard deviation of the distribution of fluxes. What is the largest outlier in the distribution, where is it on the image, and how many standard deviations is it away from the mean? 

In [ ]:
f105_flux_mean = np.mean(flux)
f105_flux_median = np.median(flux)
f105_flux_std = np.std(flux)

print(f'Mean: {f105_flux_mean:f}')
print(f'Median: {f105_flux_median:f}')
print(f'Standard deviation: {f105_flux_std:f}')

f105_flux_shifted = flux-f105_flux_mean
f105_flux_shifted = np.abs(f105_flux_shifted)
f105_largest_outlier_shifted = max(f105_flux_shifted)

for i,val in enumerate(f105_flux_shifted):
    if val == f105_largest_outlier_shifted:
        f105_largest_outlier = flux[i]
        f105_largest_outlier_index = i
f105_largest_outlier_sigma = f105_largest_outlier_shifted/f105_flux_std

print(f'Largest Outlier:\n  Object: {f105_largest_outlier_index}', 
      f'Flux: {f105_largest_outlier:f}',
      f'Sigma: {f105_largest_outlier_sigma:f} \u03c3',
      sep='\n  ')


In [ ]:
from matplotlib.patches import Arrow
fig, ax = plt.subplots()
m, s = np.mean(data_sub), np.std(data_sub)
im = ax.imshow(data_sub, interpolation='nearest', cmap='gray',
               vmin=m-s, vmax=m+s, origin='lower')

i=f105_largest_outlier_index
e = Ellipse(xy=(objects['x'][i],objects['y'][i]),
                width=10*objects['a'][i],
                height=10*objects['b'][i],
                angle=objects['theta'][i] * 180. / np.pi)
e.set_facecolor('none')
e.set_edgecolor('red')
ax.add_artist(e)

arr_x = 200
arr_y = 0
_ = ax.add_artist(Arrow(x=objects['x'][i]+arr_x,y=objects['y'][i]+arr_y,dx=-arr_x+32,dy=-arr_y, fc='red',width=100))

fig.suptitle('Location of Largest Outlier')
plt.tight_layout()

plt.show()